# Statistical Hypothesis Testing

**Course:** Data Science  
**Lesson:** 07  
**Difficulty:** Intermediate

## Learning Objectives

- Formulate null and alternative hypotheses.
- Interpret significance levels, p-values, and confidence intervals.
- Explain Type I and Type II errors.
- Check common statistical assumptions.
- Perform one-sample, independent, and paired t-tests.
- Perform chi-square and one-way ANOVA tests.
- Calculate introductory effect sizes.
- Report findings clearly and cautiously.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

## 1. Load the Dataset

In [ ]:
candidate_paths = [
    Path("../datasets/employee_statistics.csv"),
    Path("Data-Science-Course/datasets/employee_statistics.csv"),
    Path("datasets/employee_statistics.csv"),
]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Place employee_statistics.csv in Data-Science-Course/datasets/.")
df = pd.read_csv(data_path, parse_dates=["Hire_Date"])
df.head()

In [ ]:
df.shape

## 2. Population and Sample

A population is the full group of interest. A sample is a subset used to estimate or test population properties.

In [ ]:
sample_df = df.sample(n=100, random_state=42)
pd.DataFrame({
    "Dataset":["Full dataset","Random sample"],
    "Rows":[len(df),len(sample_df)],
    "Average Salary":[df["Annual_Salary_USD"].mean(),sample_df["Annual_Salary_USD"].mean()]
}).round(2)

## 3. Hypotheses and Significance

The null hypothesis (`H₀`) usually represents no difference or no association.  
The alternative hypothesis (`H₁`) represents the investigated effect.

A common significance level is `α = 0.05`.

In [ ]:
alpha = 0.05

## 4. p-values

- If `p-value < α`, reject `H₀`.
- If `p-value ≥ α`, do not reject `H₀`.

A p-value is not the probability that the null hypothesis is true.

## 5. Type I and Type II Errors

| Error | Meaning |
|---|---|
| Type I | Rejecting a true null hypothesis |
| Type II | Failing to reject a false null hypothesis |

Statistical power is the probability of detecting a real effect.

## 6. Confidence Interval for Mean Salary

In [ ]:
salary = df["Annual_Salary_USD"].dropna()
mean_salary = salary.mean()
se = stats.sem(salary)
ci = stats.t.interval(0.95, df=len(salary)-1, loc=mean_salary, scale=se)
pd.Series({"Mean":mean_salary,"Standard Error":se,"95% CI Lower":ci[0],"95% CI Upper":ci[1]}).round(2)

## 7. Normality Checks

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(salary,bins=25,edgecolor="black")
plt.title("Salary Distribution")
plt.xlabel("Annual Salary (USD)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
sm.qqplot(salary,line="45",fit=True)
plt.title("Q-Q Plot of Annual Salary")
plt.tight_layout()
plt.show()

In [ ]:
shapiro_sample = salary.sample(n=min(500,len(salary)),random_state=42)
shapiro = stats.shapiro(shapiro_sample)
pd.Series({"Statistic":shapiro.statistic,"p-value":shapiro.pvalue}).round(4)

Normality tests can detect small deviations in large samples. Use visual checks, sample size, and context together.

## 8. One-Sample t-Test

In [ ]:
reference_mean = 65000
one_sample = stats.ttest_1samp(salary,popmean=reference_mean)
pd.Series({
    "Sample Mean":salary.mean(),
    "Reference Mean":reference_mean,
    "t-statistic":one_sample.statistic,
    "p-value":one_sample.pvalue
}).round(4)

In [ ]:
print("Reject H0" if one_sample.pvalue < alpha else "Do not reject H0")

### One-Sample Cohen's d

In [ ]:
one_sample_d = (salary.mean()-reference_mean)/salary.std(ddof=1)
one_sample_d

## 9. Independent Samples t-Test

In [ ]:
gender_values = df["Gender"].dropna().unique()
if len(gender_values) < 2:
    raise ValueError("At least two gender groups are required.")
g1_name,g2_name = gender_values[:2]
g1 = df.loc[df["Gender"]==g1_name,"Annual_Salary_USD"].dropna()
g2 = df.loc[df["Gender"]==g2_name,"Annual_Salary_USD"].dropna()
pd.DataFrame({
    "Group":[g1_name,g2_name],
    "Count":[len(g1),len(g2)],
    "Mean":[g1.mean(),g2.mean()],
    "Std":[g1.std(),g2.std()]
}).round(2)

In [ ]:
levene = stats.levene(g1,g2)
equal_var = bool(levene.pvalue >= alpha)
ind_test = stats.ttest_ind(g1,g2,equal_var=equal_var)
pd.Series({
    "Levene p-value":levene.pvalue,
    "Equal Variance Assumed":equal_var,
    "t-statistic":ind_test.statistic,
    "p-value":ind_test.pvalue
}).round(4)

### Independent-Groups Cohen's d

In [ ]:
def cohens_d(x,y):
    nx,ny=len(x),len(y)
    pooled=((nx-1)*x.var(ddof=1)+(ny-1)*y.var(ddof=1))/(nx+ny-2)
    return (x.mean()-y.mean())/np.sqrt(pooled)
cohens_d(g1,g2)

## 10. Paired Samples t-Test

The dataset has no repeated measurements, so this teaching example creates a reproducible post-training score from existing performance values.

In [ ]:
before = df["Performance_Score"].dropna().sample(n=min(120,df["Performance_Score"].notna().sum()),random_state=7).reset_index(drop=True)
rng=np.random.default_rng(7)
after=before+rng.normal(loc=0.25,scale=0.50,size=len(before))
paired=stats.ttest_rel(before,after)
pd.Series({
    "Before Mean":before.mean(),
    "After Mean":after.mean(),
    "t-statistic":paired.statistic,
    "p-value":paired.pvalue
}).round(4)

In [ ]:
difference=after-before
paired_d=difference.mean()/difference.std(ddof=1)
paired_d

## 11. Chi-Square Test of Independence

In [ ]:
table=pd.crosstab(df["Department"],df["Gender"])
table

In [ ]:
chi2,p,dof,expected=stats.chi2_contingency(table)
pd.Series({"Chi-square":chi2,"Degrees of Freedom":dof,"p-value":p}).round(4)

In [ ]:
expected_table=pd.DataFrame(expected,index=table.index,columns=table.columns)
expected_table.round(2)

### Cramér's V

In [ ]:
n=table.to_numpy().sum()
cramers_v=np.sqrt(chi2/(n*(min(table.shape)-1)))
cramers_v

## 12. One-Way ANOVA

In [ ]:
salary_groups=[g["Annual_Salary_USD"].dropna().values for _,g in df.groupby("Department")]
anova=stats.f_oneway(*salary_groups)
pd.Series({"F-statistic":anova.statistic,"p-value":anova.pvalue}).round(4)

### Eta Squared

In [ ]:
grand_mean=df["Annual_Salary_USD"].mean()
ss_between=sum(len(g)*(g["Annual_Salary_USD"].mean()-grand_mean)**2 for _,g in df.groupby("Department"))
ss_total=((df["Annual_Salary_USD"]-grand_mean)**2).sum()
eta_squared=ss_between/ss_total
eta_squared

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df,x="Department",y="Annual_Salary_USD")
plt.title("Salary Distribution by Department")
plt.xticks(rotation=45,ha="right")
plt.tight_layout()
plt.show()

## 13. Choosing a Test

| Question | Typical test |
|---|---|
| One sample mean vs reference | One-sample t-test |
| Two independent means | Independent t-test |
| Two repeated measurements | Paired t-test |
| Three or more means | One-way ANOVA |
| Two categorical variables | Chi-square test |

## 14. Reporting Results

Include the research question, test, sample sizes, descriptive statistics, test statistic, p-value, effect size, conclusion, and limitations.

## 15. Common Mistakes

1. Treating significance as proof.
2. Reporting p-values without effect sizes.
3. Ignoring assumptions.
4. Using independent tests for paired data.
5. Claiming causation from observational data.
6. Interpreting non-significance as proof of no effect.
7. Ignoring multiple testing and group sizes.

## 16. Exercises

1. Test whether average weekly hours differ from 40.
2. Compare performance scores between two departments.
3. Calculate Cohen's d for that comparison.
4. Test whether education level is associated with gender.
5. Calculate Cramér's V.
6. Compare job satisfaction across departments using ANOVA.
7. Calculate eta squared.
8. Create a confidence interval for training hours.
9. Check absence-day normality.
10. Write a statistical conclusion.

In [ ]:
# Write your exercise solutions here.

## Summary

You performed common hypothesis tests, checked assumptions, calculated effect sizes, and practiced responsible interpretation.

## Next Lesson

**End-to-End Data Analysis Case Study**.